In [17]:
import pandas as pd
from sklearn import preprocessing, pipeline, ensemble, compose
import datasets
import os

paths = {
    'real': '/hdd3/sonia/data/adult.csv',
    'dgpt2': '/hdd3/sonia/be_great/ckpts/dgpt2/adult-allcol/samples.csv',
    'moe': '/hdd3/sonia/be_great/ckpts/moe/dgpt2/adult-allcol/jul21/samplesclean.csv',
    'greatdgpt2': '/hdd3/sonia/be_great/ckpts/dgpt2-greatclean.csv',
    'moegreatdgpt2': '/hdd3/sonia/be_great/ckpts/great/adult/moegreatdgpt2-aug12.csv',
    'fairgan': '/hdd3/sonia/be_great/ckpts/tabfairgan/adult-april.csv',
}
rs = 4 # random state
train_frac = 0.75

ords = ['workclass', 'education', 'marital-status', 'occupation', 
        'relationship', 'race', 'sex', 'native-country'] # MUST BE IN ORDER
nums = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week', ]
labs = ['income']

In [18]:
datadict = {k:pd.read_csv(v) for (k,v) in paths.items()}
print([k+': '+str(df.shape) for (k, df) in datadict.items()]) # print shape of each dataset

min_dataset_size = min([len(df) for df in datadict.values()])
train_size = int(min_dataset_size*train_frac)
categoriesdict = dict() # collect all unique values for each of the ordinal columns
for (k, df) in datadict.items():
    print(k)
    # sample to min dataset size, shuffle, ensure cols in order w no extra cols:
    df = df.sample(min_dataset_size, random_state=rs, ignore_index=True)[ords+nums+labs]
    # remove extra spaces around strings, eg ' dog' -> 'dog'
    df = df.map(lambda x: x.strip() if type(x) == str else x)
    for col in ords:
        categoriesdict[col] = categoriesdict.get(col, []) + df[col].unique().tolist()
    datadict[k] = {'train':df.iloc[:train_size, :],
                   'test': df.iloc[train_size:, :]}
print(f'sampled {min_dataset_size} rows from each dataset, made trainsets of {train_size} rows each')

categories = []
for col in ords:
    categories.append(list(set(categoriesdict[col])))
ordenc = preprocessing.OrdinalEncoder(categories=categories)
numenc = preprocessing.StandardScaler()
lb = preprocessing.LabelBinarizer()

['real: (48842, 15)', 'dgpt2: (5935, 15)', 'moe: (9106, 15)', 'greatdgpt2: (9815, 15)', 'moegreatdgpt2: (9997, 15)', 'fairgan: (32561, 15)']
real
dgpt2
moe
greatdgpt2
moegreatdgpt2
fairgan
sampled 5935 rows from each dataset, made trainsets of 4451 rows each


In [19]:
# make random forest sklearn pipeline
def create_pipeline(trainset):
    rfc = ensemble.RandomForestClassifier(n_estimators=10, max_depth=4, random_state=rs)
    preprocessing_pipeline = compose.ColumnTransformer([
        ("ordinal_preprocessor", ordenc, ords),
        ("numerical_preprocessor", numenc, nums),
    ])
    complete_pipeline = pipeline.Pipeline([
        ("preprocessor", preprocessing_pipeline),
        ("estimator", rfc)
    ])
    
    preprocessed_labels = lb.fit_transform(trainset[labs].values.ravel()).ravel()
    complete_pipeline.fit(trainset[ords+nums], preprocessed_labels)
    return complete_pipeline

rfdict = {}
for src in datadict.keys():
    print(src)
    rfdict[src] = create_pipeline(datadict[src]['train'])

real
dgpt2


moe
greatdgpt2
moegreatdgpt2
fairgan


In [21]:
for data in datadict.keys():
    print(data)
    labels = lb.fit_transform(datadict[data]['test'][labs])
    for model in rfdict.keys():
        score = rfdict[model].score(datadict[data]['test'][ords+nums], labels)
        print(f'{model} on {data}: \t\t\t{score}')
    print('\n')

real
real on real: 			0.8402964959568733
dgpt2 on real: 			0.8322102425876011
moe on real: 			0.7668463611859838
greatdgpt2 on real: 			0.8295148247978437
moegreatdgpt2 on real: 			0.8247978436657682
fairgan on real: 			0.8214285714285714


dgpt2
real on dgpt2: 			0.7338274932614556
dgpt2 on dgpt2: 			0.7964959568733153
moe on dgpt2: 			0.738544474393531
greatdgpt2 on dgpt2: 			0.7338274932614556
moegreatdgpt2 on dgpt2: 			0.7358490566037735
fairgan on dgpt2: 			0.7237196765498652


moe
real on moe: 			0.7681940700808625
dgpt2 on moe: 			0.7789757412398922
moe on moe: 			0.7904312668463612
greatdgpt2 on moe: 			0.7553908355795148
moegreatdgpt2 on moe: 			0.7742587601078167
fairgan on moe: 			0.7661725067385444


greatdgpt2
real on greatdgpt2: 			0.805256064690027
dgpt2 on greatdgpt2: 			0.8025606469002695
moe on greatdgpt2: 			0.6037735849056604
greatdgpt2 on greatdgpt2: 			0.8092991913746631
moegreatdgpt2 on greatdgpt2: 			0.807277628032345
fairgan on greatdgpt2: 			0.8079514824797843